# Use OpenRouter With OpenAI Agents SDK

Setup Prerequisite:

1. [Signup at OpenRouter](https://openrouter.ai/)
2. [Create an API Key](https://openrouter.ai/settings/keys)
2. Select a Free Model (you can continue as we are using a free model here)

## Free and Paid Models

The OpenRouter supports the latest DeepSeek V3 0324 and 50+ other models for free. Most of them support the defacto standard: OpenAI Chat Completion API.


If you are using a free model variant (with an ID ending in :free), then you will be limited to 20 requests per minute and 200 requests per day.

**See all Models List: https://openrouter.ai/models**

Note: OpenRouter do not charge anything extra at inference time.

## Rate Limiting and Crediting

There are a few rate limits that apply to certain types of requests, regardless of account status:

- Free limit: If you are using a free model variant (with an ID ending in :free), then you will be limited to 20 requests per minute and 200 requests per day.

If your account has a negative credit balance, you may see 402 errors, including for free models. Adding credits to put your balance above zero allows you to use those models again.

[Reference](https://openrouter.ai/docs/api-reference/limits)

## Install OpenAI Agents Dep.

In [1]:
!uv pip install -Uq openai-agents

In [2]:
import nest_asyncio
nest_asyncio.apply()

## Provider Config

In [3]:
#from google.colab import userdata
import os
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [4]:
#Reference: https://openrouter.ai/docs/quickstart

BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "google/gemini-2.0-flash-exp:free"

# Some other free models on 26th March:
# https://openrouter.ai/deepseek/deepseek-chat-v3-0324:free
# https://openrouter.ai/google/gemini-2.5-pro-exp-03-25:free

## 1. Using the OpenRouter API directly

In [5]:
import requests
import json

response = requests.post(
  url=f"{BASE_URL}/chat/completions",
  headers={
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
  },
  data=json.dumps({
    "model": MODEL,
    "messages": [
      {
        "role": "user",
        "content": "What is the meaning of life?"
      }
    ]
  })
)

print(response.json())

{'id': 'gen-1758451639-6k3vxCXfwedpX1GtkyS2', 'provider': 'Google', 'model': 'google/gemini-2.0-flash-exp:free', 'object': 'chat.completion', 'created': 1758451639, 'choices': [{'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'STOP', 'index': 0, 'message': {'role': 'assistant', 'content': "Ah, the million-dollar question! The meaning of life is one of those questions that philosophers, theologians, and individuals have wrestled with for centuries. There's no single, universally accepted answer, and that's often the beauty of it. It's a **deeply personal and subjective exploration.**\n\nHere's a breakdown of some common perspectives:\n\n*   **Philosophical Perspectives:**\n\n    *   **Nihilism:**  This view argues that life is inherently without objective meaning, purpose, or intrinsic value.\n    *   **Existentialism:**  Existentialists believe that existence precedes essence.  We are born into the world without a predetermined purpose.  It's up to each individual to

In [6]:
data = response.json()
data['choices'][0]['message']['content']

"Ah, the million-dollar question! The meaning of life is one of those questions that philosophers, theologians, and individuals have wrestled with for centuries. There's no single, universally accepted answer, and that's often the beauty of it. It's a **deeply personal and subjective exploration.**\n\nHere's a breakdown of some common perspectives:\n\n*   **Philosophical Perspectives:**\n\n    *   **Nihilism:**  This view argues that life is inherently without objective meaning, purpose, or intrinsic value.\n    *   **Existentialism:**  Existentialists believe that existence precedes essence.  We are born into the world without a predetermined purpose.  It's up to each individual to create their own meaning and values through their choices and actions.  Key concepts include freedom, responsibility, and authenticity. Thinkers like Sartre and Camus fall into this category.\n    *   **Absurdism:**  A related philosophy that focuses on the conflict between humanity's search for meaning and

## 2. Using OpenAI Agents SDK

In [7]:
import asyncio
from openai import AsyncOpenAI
from agents import Agent, OpenAIChatCompletionsModel, Runner, set_tracing_disabled

client = AsyncOpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=BASE_URL
)

set_tracing_disabled(disabled=True)

async def main():
    # This agent will use the custom LLM provider
    agent = Agent(
        name="Assistant",
        instructions="You only respond in haikus.",
        model=OpenAIChatCompletionsModel(model=MODEL, openai_client=client),
    )

    result = await Runner.run(
        agent,
        "Tell me about recursion in programming.",
    )
    print(result.final_output)


if __name__ == "__main__":
    asyncio.run(main())

A function calls self,
Solving smaller, same task,
Base case stops the flow.



# OpenRouter 404 Error Solution

## Error - No endpoints found matching
```python
NotFoundError: Error code: 404 - {'error': {'message': 'No endpoints found matching your data policy. Enable prompt training here: https://openrouter.ai/settings/privacy', 'code': 404}}
```

## Cause
This error occurs when OpenRouter API can't find endpoints matching your data policy, typically because prompt training is disabled.

## Solution

1. **Enable Prompt Training**:
   - Visit [OpenRouter Privacy Settings](https://openrouter.ai/settings/privacy)
   - Toggle ON "Prompt Training" option

2. **Re-run your code** after enabling

![OpenRouter Settings Screenshot](./openrouter.png)
*(Example: Enable prompt training in privacy settings)*

## Prevention
Keep prompt training enabled for uninterrupted API access.
```